# 🃏 Poker Arena — Evidência Científica #1 (estilo *Use a Cabeça*)

**Rode a única célula abaixo** (`Shift+Enter`). Os logs vão te explicar tudo enquanto rodam: o bot aprende poker **sozinho** e você vê a *brecha* (exploitability) despencar até ~zero — a prova de que ele chegou ao jogo **ótimo** (equilíbrio de Nash).

Roda em **CPU** no Colab, em poucos minutos. Sem GPU.

In [ ]:
# ============================================================
#  POKER ARENA - Evidencia Cientifica #1  (CELULA UNICA)
#  Logs no estilo 'Use a Cabeca': eles te explicam tudo enquanto rodam.
#  (Se a 1a execucao falhar em 'import pyspiel', rode a celula de novo:
#   o Colab acabou de instalar o pacote.)
# ============================================================

!pip install -q open_spiel==1.6.15  # pin exato; receipt de hashes transitivos ainda externo

import math
import pyspiel
from open_spiel.python.algorithms import cfr, exploitability
import matplotlib.pyplot as plt

# ---------- helpers de log amigavel ----------
def box(titulo, linhas):
    largura = max([len(titulo)] + [len(l) for l in linhas])
    print('  +' + '-' * (largura + 2) + '+')
    print('  | ' + titulo.ljust(largura) + ' |')
    print('  +' + '-' * (largura + 2) + '+')
    for l in linhas:
        print('  | ' + l.ljust(largura) + ' |')
    print('  +' + '-' * (largura + 2) + '+')

def comentario(e):
    if e > 0.3:   return 'ainda da pra explorar facil - o bot mal pegou as regras'
    if e > 0.05:  return 'opa, ta ficando esperto...'
    if e > 0.005: return 'quase imbativel!'
    return 'praticamente impossivel de explorar  [imbativel]'

def barra(e, e_start):
    lo, hi = math.log10(1e-3), math.log10(max(e_start, 1e-3) + 1e-12)
    cur = math.log10(max(e, 1e-9))
    frac = max(0.0, min(1.0, (cur - lo) / (hi - lo + 1e-12)))
    n = int(frac * 20)
    return '[' + '#' * n + '.' * (20 - n) + ']'

# ---------- a historia ----------
box('BEM-VINDO! O QUE VAMOS PROVAR AQUI', [
    'Pergunta: da pra uma IA aprender a jogar poker OTIMO sozinha,',
    '          so jogando contra si mesma, sem ninguem ensinar?',
    'Resposta: SIM - e voce vai VER acontecer, ao vivo, abaixo.',
])

box('NAO TEM PERGUNTA IDIOTA', [
    'P: O que e a tal da brecha (exploitability)?',
    'R: O quanto um adversario PERFEITO te exploraria.',
    '   Menor = melhor. ZERO = ninguem tira vantagem de voce.',
    'P: O que e CFR?',
    'R: O algoritmo que aprende minimizando arrependimento -',
    '   a mesma familia do Libratus/Pluribus (que bateram pros).',
    'P: Por que comecar pelo Kuhn Poker?',
    'R: E um poker minusculo com solucao OTIMA conhecida no papel,',
    '   entao da pra PROVAR que o bot chegou la.',
])

# ---------- treino narrado ----------
def run_cfr(apelido, game_name, n_iters, every):
    print()
    box('TREINANDO: ' + apelido, [
        'O bot vai jogar milhares de maos contra si mesmo.',
        'A cada rodada ele lembra do que se arrependeu e melhora.',
        'Acompanhe a BRECHA encolhendo (barra menor = melhor):',
    ])
    game = pyspiel.load_game(game_name)
    solver = cfr.CFRSolver(game)
    iters, conv, e_start = [], [], None
    marcos = {0.1: False, 0.01: False, 0.001: False}
    for i in range(1, n_iters + 1):
        solver.evaluate_and_update_policy()
        if i == 1 or i % every == 0:
            e = exploitability.exploitability(game, solver.average_policy())
            if e_start is None:
                e_start = e
            iters.append(i)
            conv.append(e)
            print(f'   rodada {i:>4}  brecha={e:8.5f}  {barra(e, e_start)}  {comentario(e)}')
            for m in (0.1, 0.01, 0.001):
                if not marcos[m] and e < m:
                    marcos[m] = True
                    print(f'         >>> UAU! a brecha furou {m} - salto de qualidade!')
    return solver, iters, conv

kuhn_solver,  k_it, k_ex = run_cfr('Kuhn Poker (o pequeno, pra PROVAR)', 'kuhn_poker', 200, 10)
leduc_solver, l_it, l_ex = run_cfr('Leduc Holdem (maior, pra GENERALIZAR)', 'leduc_poker', 1000, 50)

# ---------- graficos ----------
print()
print('  Desenhando os graficos da brecha caindo...')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.semilogy(k_it, k_ex, marker='o', color='#c0392b')
ax1.set_title('Kuhn: a brecha despenca ate ~0 (= OTIMO)')
ax1.set_xlabel('rodadas de treino'); ax1.set_ylabel('brecha (log)')
ax1.grid(True, which='both', alpha=0.3)
ax2.semilogy(l_it, l_ex, marker='o', color='#2980b9')
ax2.set_title('Leduc: mesma tecnica, jogo maior')
ax2.set_xlabel('rodadas de treino'); ax2.set_ylabel('brecha (log)')
ax2.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig('cfr_convergence.png', dpi=150)
plt.show()

# ---------- a estrategia que o bot inventou ----------
print()
box('ESPIE A ESTRATEGIA QUE O BOT INVENTOU (Kuhn)', [
    'Cada linha = uma situacao: sua carta + o que rolou antes.',
    'Os 2 numeros = [chance de PASSAR/desistir, chance de APOSTAR/pagar].',
    'Legenda: carta 0=Valete(fraca) 1=Dama 2=Rei(forte) | p=passou b=apostou',
    'REPARE: as vezes ele APOSTA com carta fraca. Isso e BLEFE -',
    '        ninguem ensinou; ele descobriu que blefar compensa.',
])
avg = kuhn_solver.average_policy()
for info_state, idx in sorted(avg.state_lookup.items()):
    p = avg.action_probability_array[idx]
    print(f'   situacao [{info_state:<4}]  ->  passar={p[0]:.2f}   apostar={p[1]:.2f}')

# ---------- fechamento ----------
print()
box('O QUE VOCE ACABOU DE PROVAR', [
    f'Kuhn : brecha final = {k_ex[-1]:.5f}  -> ~ZERO = chegou no OTIMO (Nash).',
    f'Leduc: brecha final = {l_ex[-1]:.5f}  -> a MESMA tecnica num jogo maior.',
    'A IA aprendeu a jogar quase-perfeito SOZINHA, so jogando consigo mesma.',
    'Graficos salvos em: cfr_convergence.png',
    'Resultado cientifico #1 da feira. Proximo: NFSP / Deep CFR / PPO.',
])